# DistilBERT Fine-tuning for Toxic Comment Classification

Single, end-to-end notebook for the DistilBERT branch of the toxic-comment
project. Run all cells top-to-bottom to reproduce **every** output the report
needs.

Pipeline:

1. Imports + seed
2. Config
3. Load data + label distribution + **train/val/test leakage check** (#13)
4. **Token length distribution** (justification for `MAX_LENGTH`) (#14)
5. Tokenize + model setup + **class-weighted Trainer** (#2)
6. Multi-epoch training with `load_best_model_at_end` on val F1 (#1)
7. Standard evaluation: binary + macro + per-class metrics (#11)
8. **Validation-set threshold tuning** + re-evaluate on test (#2)
9. Save best model, tokenizer, full metrics, and best threshold
10. **Plot training curves** (loss + val precision/recall/F1) (#7)
11. **Plot test confusion matrix at the tuned threshold**
12. **Evaluate on `new_200.csv` + export error analysis tables** (#4)

All randomness is controlled by `SEED = 42` (`set_seed` + `TrainingArguments(seed=, data_seed=)`).

## Running this notebook on Google Colab

The notebook works on both **local Jupyter** and **Google Colab** with no
manual changes — the next cell auto-detects the environment.

**One-time setup before opening this notebook on Colab:**

1. Zip the whole `6713-gp-DistilBERT/` project locally (so you keep
   `train_distilbert.ipynb`, `cleaned_data/`, etc.).
2. Upload the zip to your Google Drive, e.g. into
   `My Drive/` and unzip it there so you end up with
   `My Drive/6713-gp-DistilBERT/` containing `train_distilbert.ipynb`,
   `cleaned_data/`, `requirements.txt`, ...
3. Open `train_distilbert.ipynb` directly from Drive in Colab.
4. Switch the runtime to GPU: **Runtime -> Change runtime type -> T4 GPU**.
5. Run all cells. The next cell mounts Drive, installs the missing
   dependencies, and `cd`s into the project folder so all relative paths
   (e.g. `cleaned_data/pre-trained model/6w data`) keep working.

If your project lives at a different path on Drive, edit
`COLAB_PROJECT_ROOT` in the next cell.

In [ ]:
# --- Environment setup: works on both local Jupyter and Google Colab ---
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

# If your project lives somewhere else on Drive, change this path.
COLAB_PROJECT_ROOT = "/content/drive/MyDrive/6713-gp-DistilBERT"

if IN_COLAB:
    print("Detected Google Colab environment.")

    # Mount Google Drive so we can read cleaned_data/ and write outputs/.
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

    if not os.path.isdir(COLAB_PROJECT_ROOT):
        raise FileNotFoundError(
            f"Project folder not found at {COLAB_PROJECT_ROOT}. "
            "Upload / unzip the project to that path on your Drive, "
            "or change COLAB_PROJECT_ROOT above."
        )
    os.chdir(COLAB_PROJECT_ROOT)
    print("Working directory set to:", os.getcwd())

    # Colab already ships torch with CUDA. Only install the rest.
    pkgs = [
        "transformers==5.5.4",
        "datasets==4.8.4",
        "accelerate>=0.30",
        "scikit-learn==1.6.1",
    ]
    print("Installing:", pkgs)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True
    )

    # Sanity check
    import torch
    print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print(
            "WARNING: no GPU detected. Switch the runtime to GPU "
            "(Runtime -> Change runtime type -> T4 GPU) and re-run this cell."
        )
else:
    print("Local mode (not on Colab). Using current working directory:", os.getcwd())

In [ ]:
# %pip install -r requirements.txt

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from datasets.utils.logging import disable_progress_bar
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
disable_progress_bar()

SEED = 42
set_seed(SEED)

In [ ]:
TEXT_COLUMN = "cleaned_text"
LABEL_COLUMN = "toxic"
MODEL_NAME = "distilbert-base-uncased"

MAX_LENGTH = 128
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
USE_CLASS_WEIGHTS = True
ENCODING = "utf-8"

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "cleaned_data" / "pre-trained model" / "6w data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "distilbert_jigsaw_notebook_6w"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "val.csv"
TEST_PATH = DATA_DIR / "test.csv"
MANUAL_200_PATH = PROJECT_ROOT / "cleaned_data" / "new_200.csv"

print("Dataset directory:", DATA_DIR)
print("Output directory: ", OUTPUT_DIR)
print(f"Epochs: {NUM_EPOCHS} | LR: {LEARNING_RATE} | BS: {TRAIN_BATCH_SIZE}")
print(f"Class weighting: {USE_CLASS_WEIGHTS} | Seed: {SEED}")

In [ ]:
def load_dataframe(path: Path, encoding: str = "utf-8") -> pd.DataFrame:
    df = pd.read_csv(path, encoding=encoding)
    required_columns = {TEXT_COLUMN, LABEL_COLUMN}
    missing = required_columns.difference(df.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")

    df = df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
    df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
    df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
    df = df[df[TEXT_COLUMN] != ""]
    df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(int)

    invalid_labels = set(df[LABEL_COLUMN].unique()).difference({0, 1})
    if invalid_labels:
        raise ValueError(f"{path} contains invalid labels: {sorted(invalid_labels)}")
    return df.reset_index(drop=True)


def label_distribution(df: pd.DataFrame) -> dict:
    counts = df[LABEL_COLUMN].value_counts().sort_index()
    total = float(len(df))
    return {str(int(label)): round(float(count / total), 6) for label, count in counts.items()}


train_df = load_dataframe(TRAIN_PATH, ENCODING)
val_df = load_dataframe(VAL_PATH, ENCODING)
test_df = load_dataframe(TEST_PATH, ENCODING)

print(f"Train samples: {len(train_df)} | label ratio: {label_distribution(train_df)}")
print(f"Val samples:   {len(val_df)} | label ratio: {label_distribution(val_df)}")
print(f"Test samples:  {len(test_df)} | label ratio: {label_distribution(test_df)}")

# --- Leakage check (#13): make sure no exact-text overlap across splits ---
train_texts = set(train_df[TEXT_COLUMN])
val_texts = set(val_df[TEXT_COLUMN])
test_texts = set(test_df[TEXT_COLUMN])

leak_train_val = len(train_texts & val_texts)
leak_train_test = len(train_texts & test_texts)
leak_val_test = len(val_texts & test_texts)

print("\n[Leakage check] exact-text overlap counts:")
print(f"  train ∩ val : {leak_train_val}")
print(f"  train ∩ test: {leak_train_test}")
print(f"  val   ∩ test: {leak_val_test}")
if max(leak_train_val, leak_train_test, leak_val_test) > 0:
    print("  WARNING: overlap detected. Consider de-duplicating before re-running.")
else:
    print("  OK: no exact text overlap between splits.")

train_df.head()

In [ ]:
# --- Token length distribution (#14): justify MAX_LENGTH ---
_tok_for_len = AutoTokenizer.from_pretrained(MODEL_NAME)


def token_lengths(texts, sample_size: int = 5000) -> np.ndarray:
    if len(texts) > sample_size:
        rng = np.random.default_rng(SEED)
        idx = rng.choice(len(texts), size=sample_size, replace=False)
        texts = [texts[i] for i in idx]
    enc = _tok_for_len(list(texts), add_special_tokens=True, truncation=False)
    return np.array([len(ids) for ids in enc["input_ids"]])


lengths = token_lengths(train_df[TEXT_COLUMN].tolist(), sample_size=5000)
percentiles = [50, 75, 90, 95, 99, 100]
stats = {f"p{p}": int(np.percentile(lengths, p)) for p in percentiles}
stats["mean"] = float(round(lengths.mean(), 2))
stats["std"] = float(round(lengths.std(), 2))
stats["truncated_pct_at_max_length"] = float(round((lengths > MAX_LENGTH).mean() * 100, 2))

print("Token length stats on train (sample of 5000):")
for k, v in stats.items():
    print(f"  {k}: {v}")
print(
    f"\n=> With MAX_LENGTH={MAX_LENGTH}, only {stats['truncated_pct_at_max_length']}% "
    f"of training samples are truncated, which is acceptable."
)

(OUTPUT_DIR / "token_length_stats.json").write_text(
    json.dumps(stats, indent=2), encoding="utf-8"
)

In [ ]:
def compute_metrics(eval_pred):
    """Binary + macro + per-class metrics (#11)."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    precision_b, recall_b, f1_b, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", zero_division=0
    )
    precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    p_per, r_per, f_per, _ = precision_recall_fscore_support(
        labels, predictions, labels=[0, 1], average=None, zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision": float(precision_b),
        "recall": float(recall_b),
        "f1": float(f1_b),
        "precision_macro": float(precision_m),
        "recall_macro": float(recall_m),
        "f1_macro": float(f1_m),
        "precision_neg": float(p_per[0]),
        "recall_neg": float(r_per[0]),
        "f1_neg": float(f_per[0]),
        "precision_pos": float(p_per[1]),
        "recall_pos": float(r_per[1]),
        "f1_pos": float(f_per[1]),
    }


def to_dataset(df: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(df, preserve_index=False)


train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)
test_ds = to_dataset(test_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


def tokenize_function(batch):
    return tokenizer(batch[TEXT_COLUMN], truncation=True, max_length=MAX_LENGTH)


train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.rename_column(LABEL_COLUMN, "labels")
val_ds = val_ds.rename_column(LABEL_COLUMN, "labels")
test_ds = test_ds.rename_column(LABEL_COLUMN, "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# --- Class weights (#2) for imbalanced data ---
if USE_CLASS_WEIGHTS:
    cw = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=train_df[LABEL_COLUMN].values,
    )
    class_weights_tensor = torch.tensor(cw, dtype=torch.float32)
    print(f"Class weights (balanced): {cw.tolist()}")
else:
    class_weights_tensor = None


class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights: torch.Tensor | None = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        else:
            loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    disable_tqdm=True,
    dataloader_pin_memory=False,
    seed=SEED,
    data_seed=SEED,
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
)

print("Tokenizer and trainer are ready.")

In [ ]:
train_output = trainer.train()
train_output.metrics

In [ ]:
# Standard (argmax @ 0.5) evaluation on val + test with extended metrics (#11)
val_metrics = trainer.evaluate(eval_dataset=val_ds, metric_key_prefix="eval")
test_metrics = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix="test")

print("Validation metrics (argmax):")
for k, v in val_metrics.items():
    print(f"  {k}: {v}")

print("\nTest metrics (argmax):")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# --- Threshold tuning (#2): pick best threshold on val, apply to test ---
def softmax_pos_probs(logits: np.ndarray) -> np.ndarray:
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = e / e.sum(axis=1, keepdims=True)
    return probs[:, 1]


def metrics_at_threshold(probs: np.ndarray, labels: np.ndarray, threshold: float) -> dict:
    preds = (probs >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    return {
        "threshold": float(round(threshold, 4)),
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
    }


val_pred = trainer.predict(val_ds)
test_pred = trainer.predict(test_ds)

val_probs = softmax_pos_probs(val_pred.predictions)
test_probs = softmax_pos_probs(test_pred.predictions)
val_labels = np.array(val_pred.label_ids)
test_labels = np.array(test_pred.label_ids)

thresholds = np.arange(0.10, 0.91, 0.05)
sweep = [metrics_at_threshold(val_probs, val_labels, t) for t in thresholds]
sweep_df = pd.DataFrame(sweep)
print("Threshold sweep on validation set:")
print(sweep_df.to_string(index=False))

best_row = sweep_df.loc[sweep_df["f1"].idxmax()]
best_threshold = float(best_row["threshold"])
print(f"\nBest threshold by val F1: {best_threshold:.2f} (val F1={best_row['f1']:.4f})")

val_at_best = metrics_at_threshold(val_probs, val_labels, best_threshold)
test_at_best = metrics_at_threshold(test_probs, test_labels, best_threshold)
test_at_default = metrics_at_threshold(test_probs, test_labels, 0.5)

comparison = pd.DataFrame(
    [
        {"split": "val",  "threshold": 0.50, **{k: v for k, v in val_metrics.items() if k.endswith(("accuracy", "precision", "recall", "f1"))}},
        {"split": "val",  "threshold": best_threshold, **{k: val_at_best[k] for k in ["accuracy", "precision", "recall", "f1"]}},
        {"split": "test", "threshold": 0.50, **{k: test_at_default[k] for k in ["accuracy", "precision", "recall", "f1"]}},
        {"split": "test", "threshold": best_threshold, **{k: test_at_best[k] for k in ["accuracy", "precision", "recall", "f1"]}},
    ]
)
print("\nDefault vs tuned threshold:")
print(comparison.to_string(index=False))

print("\nTest classification report at best threshold:")
print(
    classification_report(
        test_labels,
        (test_probs >= best_threshold).astype(int),
        target_names=["non-toxic", "toxic"],
        digits=4,
        zero_division=0,
    )
)

In [ ]:
# Save best model + tokenizer + full metrics + selected threshold
best_model_dir = OUTPUT_DIR / "best_model"
best_model_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(best_model_dir))
tokenizer.save_pretrained(str(best_model_dir))
trainer.state.save_to_json(str(OUTPUT_DIR / "trainer_state.json"))

metrics = {
    "config": {
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "learning_rate": LEARNING_RATE,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "use_class_weights": USE_CLASS_WEIGHTS,
        "class_weights": cw.tolist() if USE_CLASS_WEIGHTS else None,
        "seed": SEED,
    },
    "train": train_output.metrics,
    "validation_argmax": val_metrics,
    "test_argmax": test_metrics,
    "threshold_tuning": {
        "best_threshold": best_threshold,
        "val_sweep": sweep,
        "val_at_best": val_at_best,
        "test_at_default_0.5": test_at_default,
        "test_at_best": test_at_best,
    },
}

with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

(OUTPUT_DIR / "best_threshold.json").write_text(
    json.dumps({"best_threshold": best_threshold}, indent=2), encoding="utf-8"
)

print("Saved best model to:    ", best_model_dir)
print("Saved metrics to:       ", OUTPUT_DIR / "metrics.json")
print("Saved best threshold to:", OUTPUT_DIR / "best_threshold.json")

## Plotting & extra evaluation

Everything below was previously a separate `.py` script. Now it lives in the
notebook so a single top-to-bottom run produces every artefact the report
needs.

In [ ]:
# --- Training curves (#7): loss + val precision/recall/F1 from trainer_state.json ---
state_path = OUTPUT_DIR / "trainer_state.json"
state = json.loads(state_path.read_text(encoding="utf-8"))
log_history = state.get("log_history", [])

train_steps, train_losses = [], []
eval_steps, eval_epochs = [], []
eval_losses, eval_f1, eval_precision, eval_recall = [], [], [], []

for entry in log_history:
    if "loss" in entry and "eval_loss" not in entry:
        train_steps.append(int(entry["step"]))
        train_losses.append(float(entry["loss"]))
    if "eval_loss" in entry:
        eval_steps.append(int(entry["step"]))
        eval_epochs.append(float(entry.get("epoch", 0.0)))
        eval_losses.append(float(entry["eval_loss"]))
        eval_f1.append(float(entry.get("eval_f1", 0.0)))
        eval_precision.append(float(entry.get("eval_precision", 0.0)))
        eval_recall.append(float(entry.get("eval_recall", 0.0)))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
if train_losses:
    ax.plot(train_steps, train_losses, label="train loss", linewidth=1.5)
if eval_losses:
    ax.plot(eval_steps, eval_losses, "o-", label="val loss (per epoch)",
            color="tab:red", linewidth=2, markersize=6)
    best_idx = int(min(range(len(eval_losses)), key=lambda i: eval_losses[i]))
    ax.axvline(eval_steps[best_idx], color="tab:green", linestyle="--",
               alpha=0.6, label=f"best val loss @ step {eval_steps[best_idx]}")
ax.set_xlabel("Training step")
ax.set_ylabel("Loss")
ax.set_title("DistilBERT training & validation loss")
ax.grid(True, alpha=0.3)
ax.legend()
loss_png = OUTPUT_DIR / "training_curves_loss.png"
fig.tight_layout()
fig.savefig(loss_png, dpi=300, bbox_inches="tight")
plt.show()

if eval_f1:
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    x = eval_epochs if all(e > 0 for e in eval_epochs) else eval_steps
    xlabel = "Epoch" if x is eval_epochs else "Training step"
    ax.plot(x, eval_f1, "o-", label="val F1", linewidth=2)
    ax.plot(x, eval_precision, "s--", label="val precision", linewidth=1.5)
    ax.plot(x, eval_recall, "^--", label="val recall", linewidth=1.5)
    best_idx = int(max(range(len(eval_f1)), key=lambda i: eval_f1[i]))
    ax.axvline(x[best_idx], color="tab:green", linestyle="--", alpha=0.6,
               label=f"best F1={eval_f1[best_idx]:.4f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Score")
    ax.set_title("DistilBERT validation precision / recall / F1")
    ax.grid(True, alpha=0.3)
    ax.legend()
    f1_png = OUTPUT_DIR / "training_curves_f1.png"
    fig.tight_layout()
    fig.savefig(f1_png, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {f1_png}")

print(f"Saved: {loss_png}")

In [ ]:
# --- Confusion matrix on the 6w test set at the tuned threshold ---
test_preds_tuned = (test_probs >= best_threshold).astype(int)
cm = confusion_matrix(test_labels, test_preds_tuned, labels=[0, 1])

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["Non-toxic", "Toxic"]
).plot(cmap="Blues", ax=ax, colorbar=False, values_format="d")
ax.set_title(f"DistilBERT Confusion Matrix (6w Test, threshold={best_threshold:.2f})")
plt.tight_layout()

cm_path = OUTPUT_DIR / "confusion_matrix.png"
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved confusion matrix to: {cm_path}")
print("Confusion matrix values (rows=true, cols=pred), labels=[0,1]:")
for row in cm:
    print(" ", row.tolist())

In [ ]:
# --- Evaluate on the manually annotated 200-sample set (#4) ---
def metrics_from_preds(labels: np.ndarray, preds: np.ndarray) -> dict:
    p_b, r_b, f1_b, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_per, r_per, f_per, _ = precision_recall_fscore_support(
        labels, preds, labels=[0, 1], average=None, zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(p_b),
        "recall": float(r_b),
        "f1": float(f1_b),
        "precision_macro": float(p_m),
        "recall_macro": float(r_m),
        "f1_macro": float(f1_m),
        "precision_neg": float(p_per[0]),
        "recall_neg": float(r_per[0]),
        "f1_neg": float(f_per[0]),
        "precision_pos": float(p_per[1]),
        "recall_pos": float(r_per[1]),
        "f1_pos": float(f_per[1]),
    }


def categorise_error(text: str, true: int, pred: int) -> str:
    t = text.lower()
    if true == 1 and pred == 0:
        if "?" not in text and "!" not in text and len(text.split()) < 12:
            return "false_negative_short_or_implicit"
        if any(w in t for w in ["sarcas", "irony", "joke", "lol", "haha", "/s"]):
            return "false_negative_sarcasm"
        return "false_negative_implicit_toxicity"
    if true == 0 and pred == 1:
        if any(w in t for w in ["politic", "gov", "election", "trump", "biden"]):
            return "false_positive_politically_charged"
        if any(w in t for w in ["hate", "kill", "die", "stupid", "idiot", "damn"]):
            return "false_positive_strong_words_no_intent"
        return "false_positive_other"
    return "correct"


if not MANUAL_200_PATH.exists():
    print(f"Skipped: {MANUAL_200_PATH} not found.")
else:
    manual_df = load_dataframe(MANUAL_200_PATH, ENCODING)
    print(f"Loaded {len(manual_df)} samples from {MANUAL_200_PATH}")
    print(
        "Label distribution: "
        f"non-toxic={(manual_df[LABEL_COLUMN] == 0).sum()}, "
        f"toxic={(manual_df[LABEL_COLUMN] == 1).sum()}"
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    batch_size = EVAL_BATCH_SIZE
    all_logits = []
    with torch.no_grad():
        for start in range(0, len(manual_df), batch_size):
            texts = manual_df[TEXT_COLUMN].iloc[start : start + batch_size].tolist()
            enc = tokenizer(texts, padding=True, truncation=True,
                            max_length=MAX_LENGTH, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            all_logits.append(model(**enc).logits.cpu().numpy())
    manual_logits = np.concatenate(all_logits, axis=0)
    manual_probs = softmax_pos_probs(manual_logits)
    manual_labels = manual_df[LABEL_COLUMN].to_numpy()

    manual_preds_default = (manual_probs >= 0.5).astype(int)
    manual_preds_tuned = (manual_probs >= best_threshold).astype(int)

    manual_metrics_default = metrics_from_preds(manual_labels, manual_preds_default)
    manual_metrics_tuned = metrics_from_preds(manual_labels, manual_preds_tuned)
    manual_cm_tuned = confusion_matrix(
        manual_labels, manual_preds_tuned, labels=[0, 1]
    ).tolist()

    manual_payload = {
        "model_dir": str(best_model_dir),
        "csv": str(MANUAL_200_PATH),
        "n_samples": int(len(manual_df)),
        "label_distribution": {
            "0": int((manual_labels == 0).sum()),
            "1": int((manual_labels == 1).sum()),
        },
        "metrics_at_default_0.5": manual_metrics_default,
        "metrics_at_threshold": {"threshold": float(best_threshold), **manual_metrics_tuned},
        "confusion_matrix_at_threshold": {
            "labels": [0, 1],
            "matrix": manual_cm_tuned,
        },
    }

    out_df = manual_df.copy()
    out_df["prob_toxic"] = np.round(manual_probs, 4)
    out_df["pred_default_0.5"] = manual_preds_default
    out_df["pred_tuned"] = manual_preds_tuned
    out_df["correct_tuned"] = (out_df["pred_tuned"] == manual_labels).astype(int)
    out_df["error_type"] = [
        categorise_error(t, int(y), int(p))
        for t, y, p in zip(out_df[TEXT_COLUMN], manual_labels, manual_preds_tuned)
    ]

    pred_csv = OUTPUT_DIR / "manual_200_predictions.csv"
    err_csv = OUTPUT_DIR / "manual_200_errors.csv"
    metrics_path = OUTPUT_DIR / "manual_200_metrics.json"

    out_df.to_csv(pred_csv, index=False, encoding="utf-8")
    out_df[out_df["correct_tuned"] == 0].to_csv(err_csv, index=False, encoding="utf-8")
    metrics_path.write_text(json.dumps(manual_payload, indent=2), encoding="utf-8")

    print("\n=== Metrics on manual_200 (tuned threshold) ===")
    print(
        classification_report(
            manual_labels, manual_preds_tuned,
            target_names=["non-toxic", "toxic"], digits=4, zero_division=0,
        )
    )
    print("Confusion matrix [rows=true, cols=pred] labels=[0,1]:")
    for row in manual_cm_tuned:
        print(" ", row)

    n_err = int((out_df["correct_tuned"] == 0).sum())
    print(f"\nErrors: {n_err}/{len(out_df)} samples misclassified at threshold={best_threshold}")
    if n_err:
        print("Error type breakdown:")
        print(out_df.loc[out_df["correct_tuned"] == 0, "error_type"].value_counts().to_string())

    print(f"\nSaved metrics to:     {metrics_path}")
    print(f"Saved predictions to: {pred_csv}")
    print(f"Saved errors to:      {err_csv}")

## Notes

- All paths are fixed to `cleaned_data/pre-trained model/6w data` and `outputs/distilbert_jigsaw_notebook_6w`.
- If CSV reading fails due to encoding, change `ENCODING` from `"utf-8"` to `"latin-1"`.
- Because the dataset is imbalanced (≈91:9), `f1` and the per-class `recall_pos` matter much more than `accuracy`.
- A single top-to-bottom run of this notebook produces:
  - `best_model/`, `checkpoints/`, `trainer_state.json`
  - `metrics.json` (default + tuned threshold + full val sweep)
  - `best_threshold.json`, `token_length_stats.json`
  - `training_curves_loss.png`, `training_curves_f1.png`
  - `confusion_matrix.png` (at the tuned threshold)
  - `manual_200_metrics.json`, `manual_200_predictions.csv`, `manual_200_errors.csv`